In [ ]:
# Imports

import json
import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── 1. Data Ingestion & Formatting ───────────────────────────────────────────
# Point this to your target run directory
RUN_DIR = "test_run_2026_06_14_140846" # CHANGE THIS TO YOUR LATEST RUN FOLDER
HERITAGE_FILE = f"{RUN_DIR}/heritage_data.json"

with open(HERITAGE_FILE, "r") as f:
    heritage_db = json.load(f).get("prompt_chains", {})

# Flatten the database into a Pandas DataFrame.
# Because chains survive multiple generations, we "explode" the generation list 
# so we have one row per (chain_id, generation) pair.
records = []
for chain_id, data in heritage_db.items():
    meta = data.get("metadata", {})
    fitness = data.get("fitness", 0.0)
    parents = data.get("parents", [])
    
    # Extract telemetry if available (Assuming it was logged)
    accuracy = meta.get("accuracy", 0.0)
    time_taken = meta.get("time_taken", 0.0)
    token_usage = meta.get("token_usage", 0.0)
    
    mutated = meta.get("mutated", False)
    recombination_mode = meta.get("recombination_mode", "unknown")
    
    # We use a dummy lineage score here; in a real scenario, you might want to 
    # log the lineage score at the time of evaluation into the metadata.
    lineage_score = meta.get("lineage_score", fitness * 0.8) 
    
    for gen in data.get("generation", []):
        records.append({
            "chain_id": chain_id,
            "generation": gen,
            "fitness": fitness,
            "lineage_score": lineage_score,
            "parents": parents,
            "num_parents": len(parents),
            "mutated": mutated,
            "recombination_mode": recombination_mode,
            "accuracy": accuracy,
            "time_taken": time_taken,
            "token_usage": token_usage
        })

df = pd.DataFrame(records)
print(f"Loaded {len(df)} generational records from {len(heritage_db)} unique chains.")

In [ ]:
# ## 2. Global Fitness & Lineage Dynamics
# Average and top fitness score by generation.

# %%
gen_stats = df.groupby('generation').agg(
    max_fitness=('fitness', 'max'),
    avg_fitness=('fitness', 'mean'),
    avg_lineage=('lineage_score', 'mean'),
    pop_size=('chain_id', 'count')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['max_fitness'], mode='lines+markers', name='Max Fitness', line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['avg_fitness'], mode='lines+markers', name='Avg Fitness', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['avg_lineage'], mode='lines+markers', name='Avg Lineage', line=dict(color='purple', dash='dash')))

fig.update_layout(title="Fitness & Lineage Trajectories over Generations", xaxis_title="Generation", yaxis_title="Score")
fig.show()

In [ ]:
# ## 3. Lineage Score Distribution (Interactive Slider)
# Watch how the population shifts from chaotic (wide distribution) to stable (normal distribution) over time.

# %%
# Ensure generation is sorted for the animation frame
df = df.sort_values('generation')
fig = px.histogram(
    df, x="lineage_score", animation_frame="generation", 
    range_x=[0, 1], nbins=20, color_discrete_sequence=['indigo'],
    title="Lineage Score Distribution Shift per Generation"
)
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
fig.show()

In [ ]:
# ## 4. Fitness Function Telemetry (Fine-Tuning Metrics)
# Analyzes Average Accuracy, Time, and Token usage. 
# Use this to tweak alpha/beta weights in `FitnessCalculation`.

# %%
telemetry_stats = df.groupby('generation')[['accuracy', 'time_taken', 'token_usage']].mean().reset_index()

from plotly.subplots import make_subplots
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=telemetry_stats['generation'], y=telemetry_stats['accuracy'], name="Avg Accuracy"), secondary_y=False)
fig.add_trace(go.Scatter(x=telemetry_stats['generation'], y=telemetry_stats['time_taken'], name="Avg Time (s)", line=dict(dash='dash')), secondary_y=True)
fig.update_layout(title="Fitness Components Over Time")
fig.update_yaxes(title_text="Accuracy", secondary_y=False)
fig.update_yaxes(title_text="Time (s)", secondary_y=True)
fig.show()

In [ ]:
# ## 5. Recombination Mode Efficacy
# Evaluates Average Success Rate (Offspring Fitness > Avg Parent Fitness).

# %%
# Rebuild parent relationships to calculate success
recombination_data = []
for idx, row in df[df['num_parents'] == 2].iterrows():
    p1, p2 = row['parents']
    p1_fit = heritage_db.get(p1, {}).get("fitness", 0)
    p2_fit = heritage_db.get(p2, {}).get("fitness", 0)
    
    avg_parent_fit = (p1_fit + p2_fit) / 2
    success = 1 if row['fitness'] > avg_parent_fit else 0
    
    recombination_data.append({
        "generation": row['generation'],
        "mode": row['recombination_mode'],
        "success": success,
        "fitness_gain": row['fitness'] - avg_parent_fit
    })

if recombination_data:
    rec_df = pd.DataFrame(recombination_data)
    rec_stats = rec_df.groupby('mode').agg(
        success_rate=('success', 'mean'),
        avg_fitness_gain=('fitness_gain', 'mean'),
        count=('success', 'count')
    ).reset_index()
    
    fig = px.bar(rec_stats, x='mode', y='success_rate', color='avg_fitness_gain', 
                 title="Recombination Success Rate by Pairing Mode",
                 labels={'success_rate': 'Success Rate (Offspring > Parents)', 'mode': 'Pairing Mode'})
    fig.show()
else:
    print("No valid 2-parent recombination data found (yet).")

In [ ]:
# ## 6. Semantic K-Clustering Sweep (Diversity Check)
# How diverse is the gene pool? We sweep K=2 through K=10. A high Silhouette Score at a high K means healthy diversity.

# %%
# To avoid running Ollama in the notebook, we use TF-IDF on the string representations of the chains.
# In a true setup, you could load the Ollama embeddings here.

import json
with open(f"{RUN_DIR}/id_to_promptchain.json", "r") as f:
    id_to_chain = json.load(f)

clustering_results = []
vectorizer = TfidfVectorizer(max_features=100)

for gen in sorted(df['generation'].unique()):
    gen_chain_ids = df[df['generation'] == gen]['chain_id'].tolist()
    gen_texts = [str(id_to_chain.get(cid, "")) for cid in gen_chain_ids]
    
    if len(gen_texts) < 10:
        continue # Not enough data to cluster
        
    X = vectorizer.fit_transform(gen_texts)
    
    for k in range(2, min(10, len(gen_texts))):
        kmeans = KMeans(n_clusters=k, random_state=42).fit(X)
        score = silhouette_score(X, kmeans.labels_)
        clustering_results.append({"generation": gen, "k": k, "silhouette": score})

if clustering_results:
    clust_df = pd.DataFrame(clustering_results)
    fig = px.line(clust_df, x="generation", y="silhouette", color="k", 
                  title="Semantic Diversity: Silhouette Score by K-Clusters over Generations")
    fig.show()
else:
    print("Not enough generations or population size to run K-Means.")

In [ ]:
# ## 7. The Full Genetic Tree (Directed Acyclic Graph)
# Traces the lineage of the absolute best prompt chain back to Generation 0.

# %%
best_chain_id = df.loc[df['fitness'].idxmax()]['chain_id']

# Build the Graph
G = nx.DiGraph()

# Recursive function to trace ancestors
def trace_lineage(chain_id, depth=0):
    if depth > 20: return # Safety break
    if chain_id not in heritage_db: return
    
    record = heritage_db[chain_id]
    G.add_node(chain_id, fitness=record.get("fitness", 0.0), gen=min(record.get("generation", [0])))
    
    for p in record.get("parents", []):
        G.add_edge(p, chain_id)
        trace_lineage(p, depth + 1)

# Trace from the best performing chain
trace_lineage(best_chain_id)

if len(G.nodes) > 0:
    # Use hierarchical layout
    pos = nx.spring_layout(G, k=0.5, iterations=50)
    
    # Extract attributes for plotting
    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)

    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    fig = go.Figure()
    # Edges
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines'))
    # Nodes (Colored by Fitness)
    fitness_vals = [G.nodes[n]['fitness'] for n in G.nodes()]
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers',
        marker=dict(showscale=True, colorscale='Viridis', size=10, color=fitness_vals, colorbar=dict(title="Fitness")),
        text=[f"ID: {n[:10]}...<br>Fitness: {G.nodes[n]['fitness']:.3f}" for n in G.nodes()], hoverinfo='text'
    ))
    
    fig.update_layout(title="Genetic Lineage of the Ultimate Prompt Chain", showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
    fig.show()
else:
    print("Could not build genetic tree.")